<a href="https://colab.research.google.com/github/fidlarsyn/Introduction-Machine-Learning-with-python/blob/main/BAB_4_Representing_Data_and_Engineering_Features.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Penanganan Variabel Kategorikal (One-Hot-Encoding)**
Variabel kategorikal harus diubah menjadi bentuk numerik agar algoritma machine learning dapat memprosesnya. Masalah umum muncul ketika kategori direpresentasikan oleh angka (seperti kode pos atau ID kategori), karena pandas.get_dummies secara default hanya memproses kolom bertipe object atau category.

In [ ]:
# Membuat dataset contoh dengan kategori numerik
demo_df = pd.DataFrame({'Nilai Kontinu': [10, 20, 30],
                        'Kategori Angka': [0, 1, 0]})

# Pro-Tip: get_dummies akan mengabaikan 'Kategori Angka' jika tipenya masih integer.
# Kita harus mengonversinya ke string agar dianggap sebagai kategori.
demo_df['Kategori Angka'] = demo_df['Kategori Angka'].astype(str)

# Menerapkan One-Hot-Encoding
display(pd.get_dummies(demo_df))

# Penjelasan Ahli:
# Menggunakan One-Hot-Encoding mencegah model menganggap kategori '1' lebih besar
# nilainya daripada kategori '0', yang dapat menyesatkan model linear.

# **Binning, Diskretisasi, dan Perbandingan Model**
Binning (diskretisasi) mengubah fitur kontinu menjadi fitur kategorikal untuk membantu model linear menangkap pola nonlinear. Berikut adalah perbandingan performa antara Regresi Linear dan Pohon Keputusan sebelum dan sesudah binning.

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.preprocessing import OneHotEncoder

# Pembuatan data sintetis
X, y = mglearn.datasets.make_wave(n_samples=100)
line = np.linspace(-3, 3, 1000, endpoint=False).reshape(-1, 1)

# Membuat 10 bin secara merata antara -3 dan 3
bins = np.linspace(-3, 3, 11)
which_bin = np.digitize(X, bins=bins)

# Transformasi bin ke One-Hot-Encoding
encoder = OneHotEncoder(sparse=False)
X_binned = encoder.fit_transform(which_bin)
line_binned = encoder.transform(np.digitize(line, bins=bins))

# Perbandingan Model
# 1. Model pada data asli
reg_orig = LinearRegression().fit(X, y)
tree_orig = DecisionTreeRegressor(min_samples_split=3).fit(X, y)

# 2. Model pada data binned
reg_bin = LinearRegression().fit(X_binned, y)
tree_bin = DecisionTreeRegressor(min_samples_split=3).fit(X_binned, y)

# Visualisasi dan Skor
print(f"Skor R^2 Linear Regression (Asli): {reg_orig.score(X, y):.2f}")
print(f"Skor R^2 Linear Regression (Binned): {reg_bin.score(X_binned, y):.2f}")
print(f"Skor R^2 Decision Tree (Asli): {tree_orig.score(X, y):.2f}")

# Catatan Pakar: Binning sangat menguntungkan model linear karena memberinya
# kemampuan untuk memiliki prediksi berbeda di setiap interval (tangga),
# namun hampir tidak berdampak pada Decision Tree yang sudah memiliki sifat membagi data.

# **Fitur Interaksi dan Polinomial**
Cara lain untuk memperkaya model linear adalah dengan menambahkan fitur polinomial (kuadrat, kubik) atau interaksi antar fitur.

In [ ]:
from sklearn.preprocessing import PolynomialFeatures
from mglearn.datasets import load_extended_boston

# Memuat dataset Boston Housing yang sudah diperluas
X, y = load_extended_boston()

# Menambahkan fitur polinomial derajat 2 (termasuk interaksi antar fitur)
poly = PolynomialFeatures(degree=2, include_bias=False)
X_poly = poly.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(X_poly, y, random_state=0)

# Uji pada model linear (Ridge)
from sklearn.linear_model import Ridge
ridge = Ridge().fit(X_train, y_train)
print(f"Skor Ridge dengan Polinomial: {ridge.score(X_test, y_test):.2f}")

# Pro-Tip: Fitur polinomial sangat membantu model linear, tetapi pada model yang
# sudah kompleks seperti Random Forest, fitur ini seringkali tidak menambah performa
# dan justru memperlambat komputasi.

# **Transformasi Nonlinear Univariat**
Banyak model statistik bekerja paling baik jika data terdistribusi normal. Untuk data yang miring (skewed), transformasi logaritma sangat efektif.

In [ ]:
import numpy as np

# Membuat data acak dengan distribusi miring (Poisson-like)
rnd = np.random.RandomState(0)
X = rnd.exponential(scale=2, size=(1000, 1))
y = rnd.normal(size=(1000,))

# Transformasi logaritma (ditambah 1 untuk menangani nilai 0)
X_log = np.log(X + 1)

# Visualisasi Dampak
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(X, bins=25, color='gray')
axes[0].set_title("Distribusi Fitur Asli (Miring)")
axes[1].hist(X_log, bins=25, color='blue')
axes[1].set_title("Distribusi Setelah Log(X+1) (Lebih Normal)")
plt.show()

# **Seleksi Fitur Otomatis (Automatic Feature Selection)**
Terlalu banyak fitur dapat menyebabkan overfitting. Kita menggunakan tiga metode utama untuk menyeleksi fitur yang paling relevan.

# Statistik Univariat (SelectPercentile)
Metode ini menguji hubungan antara setiap fitur dan target secara individual.

In [ ]:
from sklearn.feature_selection import SelectPercentile, f_classif
from sklearn.datasets import load_breast_cancer

cancer = load_breast_cancer()
# Menambahkan 50 fitur sampah (noise) untuk menguji seleksi
rng = np.random.RandomState(42)
noise = rng.normal(size=(len(cancer.data), 50))
X_w_noise = np.hstack([cancer.data, noise])

X_train, X_test, y_train, y_test = train_test_split(X_w_noise, cancer.target, random_state=42, test_size=.5)

select = SelectPercentile(score_func=f_classif, percentile=50)
select.fit(X_train, y_train)
X_train_selected = select.transform(X_train)

# Visualisasi fitur yang disimpan (get_support)
mask = select.get_support()
plt.matshow(mask.reshape(1, -1), cmap='gray_r')
plt.xlabel("Indeks Fitur (Hitam = Terpilih)")
plt.yticks([])
plt.show()

# Seleksi Berbasis Model (SelectFromModel)
Menggunakan model lain (misalnya Random Forest) untuk menentukan kepentingan fitur.

In [ ]:
from sklearn.feature_selection import SelectFromModel
from sklearn.ensemble import RandomForestClassifier

select = SelectFromModel(
    RandomForestClassifier(n_estimators=100, random_state=42),
    threshold="median"
)
select.fit(X_train, y_train)
X_train_l1 = select.transform(X_train)

# Pro-Tip: Seleksi berbasis model lebih kuat karena mempertimbangkan
# interaksi antar fitur, tidak seperti statistik univariat.

# Seleksi Fitur Iteratif (RFE)
Menghapus fitur secara bertahap hingga jumlah yang diinginkan tercapai.

In [ ]:
from sklearn.feature_selection import RFE

select = RFE(RandomForestClassifier(n_estimators=100, random_state=42),
             n_features_to_select=40)
select.fit(X_train, y_train)
X_train_rfe = select.transform(X_train)

# **Pemanfaatan Pengetahuan Pakar (Studi Kasus: Citi Bike)**
Dalam data runtun waktu (time series), representasi fitur adalah segalanya. Untuk data peminjaman sepeda, jam 23:00 dan jam 00:00 sebenarnya berdekatan, namun jika dianggap sebagai angka kontinu, model linear akan melihatnya sangat jauh.

In [ ]:
# Ekstraksi fitur waktu dari index datetime
# Misalkan 'citibike' adalah data dengan index datetime
# X_hour = citibike.index.hour.values.reshape(-1, 1)

# Rekayasa Fitur Pakar:
# Gunakan OneHotEncoder pada fitur jam, bukan menganggapnya integer.
# Ini memungkinkan model linear menangkap pola sibuk di jam tertentu
# tanpa dipaksa mengikuti tren linear naik/turun sepanjang hari.
enc = OneHotEncoder()
# X_hour_onehot = enc.fit_transform(X_hour)

# Pro-Tip: Menambahkan fitur "Apakah Akhir Pekan?" seringkali lebih
# krusial bagi akurasi model daripada algoritma yang sangat kompleks.